<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-03-prompting/lesson-3.1-system-prompts/practice/GCP_Capstone_3.1_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 3.1 — System Prompts & Generation Config

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the unified `google-genai` SDK plus `scipy` (used for the A/B t-test in Exercise 7), authenticate with Application Default Credentials, and initialise the Vertex client. Run this cell first — every exercise below depends on `client`, `genai`, and `types`.

In [ ]:
%%bash
pip install -q google-genai scipy

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
LOCATION = 'us-central1'          # 'asia-south1' for India production

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)
print('Client ready:', PROJECT_ID, LOCATION)

## Exercise 1: First System Prompt

**Difficulty:** Easy

Write a system prompt that makes Gemini respond as a pirate. Test with 3 questions.

1. Set system_instruction inside GenerateContentConfig
2. Test 3 questions
3. Verify pirate speak

In [ ]:
# The persona lives in system_instruction. A chat session keeps it alive
# across every turn, so all 3 questions get the pirate treatment.
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        system_instruction='You are a pirate. Respond in pirate speak.',
        temperature=0.7,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

for q in ['What is Python?', 'And Django?', 'What is a Kubernetes pod?']:
    r = chat.send_message(q)
    print(f'Q: {q}\nA: {r.text[:150]}\n')

## Exercise 2: Temperature Sweep

**Difficulty:** Easy

Run the same prompt at 0.0, 0.3, 0.7, 1.0, 1.5. Compare outputs.

1. Use: "Suggest a name for an AI assistant"
2. Test at 5 temperatures
3. Observe diversity increases with temperature

In [ ]:
prompt = 'Suggest a name for an AI research assistant.'

for temp in [0.0, 0.3, 0.7, 1.0, 1.5]:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temp, max_output_tokens=50,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    print(f'  temp={temp:.1f} | {r.text.strip()[:60]}')

## Exercise 3: ThinkingConfig Comparison

**Difficulty:** Easy

Test a math question with budget=0, 1024, 8192. Compare tokens and accuracy.

1. Ask: "What is 17 * 23 + 456 - 89?"
2. Print thinking tokens for each budget
3. Check if accuracy improves with more thinking

In [ ]:
question = 'What is 17 * 23 + 456 - 89?'  # correct answer: 758

for budget in [0, 1024, 8192]:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=question,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=budget),
            max_output_tokens=1000),
    )
    think_tok = r.usage_metadata.thoughts_token_count or 0
    out_tok = r.usage_metadata.candidates_token_count or 0
    print(f'  budget={budget:<6} think={think_tok:<5} out={out_tok:<5} answer={r.text.strip()[:40]}')

## Exercise 4: 5-Section Code Reviewer

**Difficulty:** Medium

Design a code reviewer persona with role/instructions/constraints/format/guardrails.

1. Define all 5 sections with XML tags
2. Test with a Python snippet
3. Verify structured output with severity counts

In [ ]:
# Same 5-section XML template as the RESEARCH_ANALYST persona in the lesson,
# re-targeted to code review. Each tag isolates one concern for the model.
CODE_REVIEWER = """
<role>
You are ReviewBot, a senior Python engineer who has reviewed 10,000+ PRs.
</role>

<instructions>
1. Read the submitted code and identify every issue.
2. Classify each issue as Critical, Major, or Minor.
3. Suggest a concrete fix for each issue.
4. Note good practices you see.
</instructions>

<constraints>
- Be direct but respectful.
- Reference specific line numbers or symbols.
- Do not rewrite the whole file; point to fixes.
</constraints>

<output_format>
## Summary
[counts: X Critical, Y Major, Z Minor]
## Critical
[...]
## Major
[...]
## Minor
[...]
## Good Practices
[...]
</output_format>

<guardrails>
- Do NOT invent issues to pad the review.
- If the code is clean, say so explicitly.
</guardrails>
"""

snippet = '''
def div(a, b):
    result = a / b
    return result

password = "admin123"
for i in range(len(data)):
    print(data[i])
'''

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=f'Review this Python code:\n```python\n{snippet}\n```',
    config=types.GenerateContentConfig(
        system_instruction=CODE_REVIEWER,
        temperature=0.2, max_output_tokens=1200,
        thinking_config=types.ThinkingConfig(thinking_budget=1024)),
)
print(r.text)

## Exercise 5: Bilingual Hindi/English Persona

**Difficulty:** Medium

Create a bilingual persona. Test with Hindi, English, and Hinglish.

1. Define persona with language detection rules
2. Test 3 messages in different languages
3. Verify correct language in responses

In [ ]:
BILINGUAL = """
<role>You are SahayakBot, a bilingual Hindi-English support agent.</role>
<instructions>
- Detect language from user message
- Hindi -> Devanagari with English tech terms
- Use formal 'aap' form. Currency in INR.
</instructions>
"""

for msg in ['What is UPI?', 'UPI kya hai?', 'mujhe payment issue hai']:
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=msg,
        config=types.GenerateContentConfig(
            system_instruction=BILINGUAL, temperature=0.5,
            max_output_tokens=300,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    print(f'Q: {msg}\nA: {r.text[:150]}...\n')

## Exercise 6: RAG + Persona Integration

**Difficulty:** Medium

Combine retrieval (the Firestore RAG from Lesson 2.3) with a RAG_CONFIG persona. Compare grounding.

1. Retrieve context with FirestoreRAG.search()
2. Generate WITHOUT system prompt
3. Generate WITH RAG system prompt
4. Compare hallucination rates

In [ ]:
# In the full capstone you would call FirestoreRAG.search() from Lesson 2.3.
# Here we stand in a fixed retrieved passage so the notebook runs standalone;
# swap `context` for `rag.search(query)` results when the RAG index is live.
context = (
    'DocuMind Pro costs 4,999 INR per month for up to 5 users. '
    'It supports PDF and DOCX uploads. It does NOT support video files.'
)
query = 'How much does DocuMind Pro cost, and does it support video uploads?'

# (2) Bare generation — no persona, no grounding rule
bare = client.models.generate_content(
    model='gemini-3.6-flash', contents=query,
    config=types.GenerateContentConfig(
        temperature=0.7,
        thinking_config=types.ThinkingConfig(thinking_budget=0)),
)
print('WITHOUT system prompt:\n', bare.text[:250], '\n')

# (3) Grounded generation — RAG persona forces answers from context only
RAG_CONFIG = types.GenerateContentConfig(
    system_instruction='Answer ONLY from the provided context. Cite sources. '
                       'If the context does not contain the answer, say so.',
    temperature=0.1, top_p=0.9, max_output_tokens=2048,
    thinking_config=types.ThinkingConfig(thinking_budget=0))

grounded = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=f'Context:\n{context}\n\nQuestion: {query}',
    config=RAG_CONFIG,
)
print('WITH RAG system prompt:\n', grounded.text[:250])

## Exercise 7: A/B Test Framework

**Difficulty:** Challenge

Build ab_test_prompts() with a t-test. Test 2 personas across 5 queries with statistical comparison.

1. Define 2 variants with identical temperature
2. 5 test cases with keywords
3. 5 runs per case (50 total calls)
4. Compute t-test p-value

In [ ]:
import time, statistics
from scipy import stats

def ab_test(client, variants, test_cases, runs=5):
    results = {}
    for name, sys_p, temp in variants:
        results[name] = []
        for tc_in, kws in test_cases:
            for _ in range(runs):
                start = time.time()
                r = client.models.generate_content(
                    model='gemini-3.6-flash', contents=tc_in,
                    config=types.GenerateContentConfig(
                        system_instruction=sys_p, temperature=temp,
                        max_output_tokens=500, seed=42,
                        thinking_config=types.ThinkingConfig(thinking_budget=0)))
                lat = (time.time()-start)*1000
                txt = r.text or ''
                recall = sum(1 for k in kws if k.lower() in txt.lower())/len(kws)
                results[name].append({'recall':recall,'latency':lat})

    for name, data in results.items():
        recalls = [d['recall'] for d in data]
        print(f'{name}: recall={statistics.mean(recalls):.3f} +/- {statistics.stdev(recalls):.3f}')

    names = list(results.keys())
    if len(names) >= 2:
        a = [d['recall'] for d in results[names[0]]]
        b = [d['recall'] for d in results[names[1]]]
        _, p = stats.ttest_ind(a, b)
        print(f'p-value: {p:.4f} | Significant: {"YES" if p<0.05 else "NO"}')

variants = [
    ('concise', 'Be concise and factual.', 0.3),
    ('expert', 'You are a senior GCP architect. Give detailed answers.', 0.3),
]
tests = [
    ('What is Cloud Run?', ['serverless','container','scale']),
    ('What is a VPC?', ['network','virtual','private']),
    ('What is Cloud Storage?', ['object','bucket','durable']),
    ('What is BigQuery?', ['analytics','warehouse','sql']),
    ('What is Pub/Sub?', ['message','publish','subscribe']),
]
ab_test(client, variants, tests)  # 2 variants x 5 cases x 5 runs = 50 calls

## Exercise 8: prompt_config.py Module

**Difficulty:** Challenge

Build 4 configs (RAG / Analysis / Creative / Code) plus a get_config() router.

1. RAG: temp=0.1, thinking=0
2. Analysis: temp=0.3, thinking=4096
3. Creative: temp=0.8, thinking=1024
4. Code: temp=0.0, thinking=4096
5. Test all 4 with appropriate queries

In [ ]:
# DocuMind Prompt Config Module — the presets you would drop into prompt_config.py
RAG_CONFIG = types.GenerateContentConfig(
    system_instruction='Answer ONLY from provided context. Cite sources.',
    temperature=0.1, top_p=0.9, max_output_tokens=2048,
    thinking_config=types.ThinkingConfig(thinking_budget=0))

ANALYSIS_CONFIG = types.GenerateContentConfig(
    system_instruction='Compare and synthesize across documents. Rate confidence.',
    temperature=0.3, top_p=0.95, max_output_tokens=4096,
    thinking_config=types.ThinkingConfig(thinking_budget=4096))

CREATIVE_CONFIG = types.GenerateContentConfig(
    system_instruction='Be imaginative and varied. Offer multiple distinct ideas.',
    temperature=0.8, top_p=0.95, max_output_tokens=2048,
    thinking_config=types.ThinkingConfig(thinking_budget=1024))

CODE_CONFIG = types.GenerateContentConfig(
    system_instruction='Write clean Python with type hints and error handling.',
    temperature=0.0, max_output_tokens=8192,
    thinking_config=types.ThinkingConfig(thinking_budget=4096))

CONFIGS = {
    'rag': RAG_CONFIG,
    'analysis': ANALYSIS_CONFIG,
    'creative': CREATIVE_CONFIG,
    'code': CODE_CONFIG,
}

def get_config(task: str) -> types.GenerateContentConfig:
    """Route a task name to its tuned config; default to RAG (safest/grounded)."""
    return CONFIGS.get(task.lower(), RAG_CONFIG)

# Test each preset with a query that suits it
probes = {
    'rag': 'Summarise the refund policy from the docs.',
    'analysis': 'Compare RAG vs fine-tuning for our support bot.',
    'creative': 'Suggest 3 taglines for DocuMind.',
    'code': 'Write a function to chunk text into 500-token windows.',
}
for task, q in probes.items():
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=q, config=get_config(task))
    print(f'[{task}] {r.text[:80]}...')